# State CPI and weekly fuel prices

This notebook combines only two verified official datasets: `cpi_state` from DOSM and `fuelprice` from the Ministry of Finance. CPI values are indices (base 2010 = 100), while fuel prices are RM per litre. They are separate measures and are not combined into a household-cost estimate.

Sources: [state CPI metadata](https://open.dosm.gov.my/data-catalogue/cpi_state), [fuel-price metadata](https://data.gov.my/data-catalogue/fuelprice), [OpenDOSM API](https://api.data.gov.my/opendosm), and [Data Catalogue API](https://api.data.gov.my/data-catalogue). Both dataset pages state CC BY 4.0 licensing. The official API needs no token; each API family allows four requests per minute. Price history includes subsidized and geographic variants; use the corresponding fields and caveats from the metadata page.

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

OPEN_API = "https://api.data.gov.my/opendosm"
CATALOGUE_API = "https://api.data.gov.my/data-catalogue"

state_response = requests.get(
    OPEN_API,
    params={"id": "cpi_state", "filter": "overall@division", "date_start": "2021-01-01@date", "sort": "date", "limit": 2000, "meta": "true"},
    timeout=30,
)
state_response.raise_for_status()
state_payload = state_response.json()
states = pd.DataFrame(state_payload["data"])
states["date"] = pd.to_datetime(states["date"])

fuel_response = requests.get(
    CATALOGUE_API,
    params={"id": "fuelprice", "filter": "level@series_type", "sort": "date", "limit": 1000, "meta": "true"},
    timeout=30,
)
fuel_response.raise_for_status()
fuel_payload = fuel_response.json()
fuel = pd.DataFrame(fuel_payload["data"])
fuel["date"] = pd.to_datetime(fuel["date"])

print("State CPI as of:", state_payload["meta"].get("data_as_of"), "rows:", len(states))
print("Fuel prices as of:", fuel_payload["meta"].get("data_as_of"), "rows:", len(fuel))
print("State CPI coverage:", states["date"].min().date(), "to", states["date"].max().date())
print("Fuel-price coverage:", fuel["date"].min().date(), "to", fuel["date"].max().date())

## Compare the latest available state indices

The comparison uses each state's overall index for the latest date present in the returned dataset. It ranks index values; it does not rank the RM cost of living.

In [ ]:
latest_state_date = states["date"].max()
latest_states = (
    states.loc[states["date"] == latest_state_date, ["state", "index"]]
    .sort_values("index", ascending=False)
)
display(latest_states)
ax = latest_states.sort_values("index").plot.barh(
    x="state", y="index", figsize=(9, 6), legend=False, color="#5579b8"
)
ax.set_title(f"Overall CPI by state — {latest_state_date:%B %Y}")
ax.set_xlabel("Index (2010 = 100)")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

## Plot weekly retail fuel prices

`ron95`, `ron97`, and `diesel` are the source's general-market petrol and Peninsular Malaysia diesel values. Other fields, including `diesel_eastmsia`, `ron95_budi95`, and `ron95_skps`, cover different geographies or eligibility rules; consult the [official fuel-price caveats](https://data.gov.my/data-catalogue/fuelprice) before comparing them.

In [ ]:
fuel_columns = [column for column in ["ron95", "ron97", "diesel"] if column in fuel]
ax = fuel.plot(x="date", y=fuel_columns, figsize=(11, 4), linewidth=1.5)
ax.set_title("Weekly retail fuel prices (RM per litre)")
ax.set_xlabel("Effective date")
ax.set_ylabel("RM / litre")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()
display(fuel[["date", *fuel_columns]].tail())